# 🚀 Training Mini-LM v4 di Kaggle — backprop LENGKAP (terverifikasi)

v4 = perbaikan tegas v3: backward attention & FFN benar-benar belajar
(gradien-check lolos di 179 param). Model sekarang BISA melatih mental
sendiri — ini training pertama v4 di cloud.

**Alur:** tarik arsitektur+korpus → BPE → latih v4 (d lebih besar, iterasi tinggi)
→ loss+perplexity tercatat → checkpoint → generate → simpan → push balik HF.

In [ ]:
# (1) Siapkan arsitektur & korpus: HF dulu, fallback /kaggle/input
!pip -q install numpy huggingface_hub 2>&1 | tail -1
import os, sys, json
os.makedirs('/kaggle/working/work', exist_ok=True)
os.chdir('/kaggle/working/work')

hf_ok = False
try:
    os.makedirs('model/aksara_ai', exist_ok=True)
    from huggingface_hub import hf_hub_download, snapshot_download
    snapshot_download(repo_id='Jokoboy/aksara-model',
                      allow_patterns=['aksara_ai/*'],
                      local_dir='model')
    hf_hub_download(repo_id='Jokoboy/aksara-corpus',
                    filename='korpus_besar.txt', local_dir='corpus')
    hf_ok = os.path.exists('model/aksara_ai/model_v4.py')
    print('HF pull OK + korpus_besar.txt')
except Exception as e:
    print('HF gagal, coba /kaggle/input:', str(e)[:80])

# fallback: dataset manual (Upload via menu Add Data)
if not hf_ok:
    import glob
    inputs = glob.glob('/kaggle/input/*/*')
    print('input candidates:', inputs[:6])
    # cari korpus txt + folder berisi model_v4.py
    korpus = None
    ai_dir = None
    for p in inputs:
        if p.endswith('.txt') and os.path.getsize(p) > 100_000:
            korpus = p
        if p.endswith('model_v4.py'):
            ai_dir = os.path.dirname(p)
    if korpus and ai_dir:
        os.makedirs('corpus', exist_ok=True)
        os.makedirs('model/aksara_ai', exist_ok=True)
        import shutil
        shutil.copy(korpus, 'corpus/korpus_besar.txt')
        for f in os.listdir(ai_dir):
            if f.endswith('.py'):
                shutil.copy(os.path.join(ai_dir, f), 'model/aksara_ai/'+f)
        hf_ok = True
        print('Pakai /kaggle/input OK')
print('READY:', hf_ok)

In [ ]:
# (2) Bungkus jadi package aksara/ai biar import relatif jalan
import os, shutil, sys
pkg = '/kaggle/working/work/aksara'
os.makedirs(pkg + '/ai', exist_ok=True)
open(pkg + '/__init__.py', 'w').write('')
open(pkg + '/ai/__init__.py', 'w').write('')
for f in os.listdir('/kaggle/working/work/model/aksara_ai'):
    if f.endswith('.py'):
        shutil.copy('/kaggle/working/work/model/aksara_ai/'+f, pkg + '/ai/'+f)
sys.path.insert(0, pkg)
sys.path.insert(0, '/kaggle/working/work')
import aksara.ai.model_v4 as m4
print('model_v4 import OK')

In [ ]:
# (3) Korpus + hardware
import os, time, subprocess
korpus = open('/kaggle/working/work/corpus/korpus_besar.txt', encoding='utf-8').read()
print(f'Korpus: {len(korpus):,} karakter')
print('CPU:', os.cpu_count())
gpu = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True)
print('GPU:', gpu.stdout.strip() or 'CPU only')
# catatan: loop v4 numpy sequential — GPU tak dipakai numpy, tp Kaggle beri RAM+uptime stabil

In [ ]:
# (4) TRAINING v4 — BPE + backward lengkap + loss/perplexity + checkpoint
from aksara.ai.tokenizer_bpe import latih as bpe_latih, ubah_ke_id, ubah_ke_teks

t0 = time.time()
bpe = bpe_latih(korpus, ukuran_vokab=6000)
ids = ubah_ke_id(bpe, korpus)
print(f'BPE vocab 6000: {len(ids):,} token ({time.time()-t0:.0f}s)')

# hold-out terakhir 5% buat eval perplexity
n_uji = max(2000, len(ids)//20)
seq_train, seq_uji = ids[:-n_uji], ids[-n_uji:]

os.makedirs('/kaggle/working/work/ckpt', exist_ok=True)
t0 = time.time()
hasil = m4.latih_v4(seq_train[:200000],
                    d=192, kepala=8, lapisan=4, E=4, blok=64,
                    iterasi=4000, laju=3e-3, benih=0,
                    seq_uji=seq_uji, cekpoint=1000,
                    jalan_cekpoint='/kaggle/working/work/ckpt')
print(f'Training 200k token, 4000 iter, d192/L4/MoE4: {time.time()-t0:.0f}s')
print('best_loss:', hasil['best_loss'], '| perplexity eval:', hasil['perplexity'])
model = hasil['bobot']

In [ ]:
# (5) Generate & simpan (model + tokenizer)
for seed in ["bahasa indonesia", "keamanan aplikasi", "program python", "harga"]:
    s = ubah_ke_id(bpe, seed)[:5]
    try:
        out = m4.tulis_v4(model, s, 24, 0.6, 0)
        print(f"[{seed}] -> {ubah_ke_teks(bpe, out)[:90]}")
    except Exception as e:
        print(f"[{seed}] err", str(e)[:60])

import pickle
with open('/kaggle/working/model_v4_kaggle.aksm','wb') as f:
    pickle.dump({'bobot': model, 'bpe': bpe.meta if hasattr(bpe,'meta') else bpe}, f)
print(len(open('/kaggle/working/model_v4_kaggle.aksm','rb').read()), 'byte tersimpan')
from IPython.display import FileLink
FileLink('model_v4_kaggle.aksm')

In [ ]:
# (6) Push balik HF (opsional). Token HF lo di Settings > Access Tokens > Write
HF_TOKEN = ""   # isi token lo (atau kosong biar skip)
if HF_TOKEN:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.upload_file(path_or_fileobj=open('/kaggle/working/model_v4_kaggle.aksm','rb'),
                    path_in_repo='model_v4_kaggle.aksm', repo_id='Jokoboy/aksara-model')
    print('PUSH KE HF SELESAI')
else:
    print('(lewati) biar push, isi HF_TOKEN cell ini')

## Setelah selesai
- **best_loss** vs v3 (yang frozen, nggak turun beneran): v4 harus jauh lebih rendah.
- Simpan **model_v4_kaggle.aksm** → buka di session Aksara: `model_v4.ak` dukung `baca_serial`.
- Kalau loss stagnan: naikkan data (jalankan `ekspansi_korpus.py --kategori ... --target 5000000`), bukan cuma param.
- Semua dicatat di `~/.huntingx/` memori (jurnal aksara-minilm).